<a href="https://colab.research.google.com/github/codeyson/energy_demand_forecasting/blob/main/energy_demand_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas numpy scikit-learn torch

In [ ]:
import pandas as pd

df = pd.read_csv("energy_dataset.csv")

# parse time
df['time'] = pd.to_datetime(df['time'], utc=True, errors='coerce')

# sort
df = df.sort_values('time')

#lag features
df['lag_1'] = df['total load actual'].shift(1)
df['lag_24'] = df['total load actual'].shift(24)

# Fill missing values
df = df.fillna(method='ffill')

### Feature Engineering

In [ ]:
# Time features
df['hour'] = df['time'].dt.hour
df['dayofweek'] = df['time'].dt.dayofweek
df['month'] = df['time'].dt.month

In [ ]:
features = [
    'generation solar',
    'generation wind onshore',
    'generation wind offshore',
    'generation nuclear',
    'generation fossil gas',
    'hour', 'dayofweek', 'month'
]

target = 'total load actual'

df = df[features + [target]]

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df)

In [ ]:
import numpy as np

def create_sequences(data, seq_length=48):
    X, y = [], []

    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length, :-1])  # features
        y.append(data[i+seq_length, -1])     # target

    return np.array(X), np.array(y)

X, y = create_sequences(data_scaled, seq_length=48)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False   # 👈 critical
)

In [ ]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, droput=0.2, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]   # last time step
        out = self.fc(out)
        return out

In [ ]:
model = LSTMModel(input_size=X.shape[2])

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

for epoch in range(20):
    model.train()

    output = model(X_train_t)
    loss = criterion(output, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 0.20848162472248077
Epoch 1, Loss: 0.1764352023601532
Epoch 2, Loss: 0.147220179438591
Epoch 3, Loss: 0.12047392129898071
Epoch 4, Loss: 0.09611447900533676
Epoch 5, Loss: 0.07444863021373749
Epoch 6, Loss: 0.056425876915454865
Epoch 7, Loss: 0.044067349284887314
Epoch 8, Loss: 0.04108690842986107
Epoch 9, Loss: 0.04977702349424362
Epoch 10, Loss: 0.057050514966249466
Epoch 11, Loss: 0.0552101731300354
Epoch 12, Loss: 0.04908793792128563
Epoch 13, Loss: 0.043371669948101044
Epoch 14, Loss: 0.03975467383861542
Epoch 15, Loss: 0.03814587742090225
Epoch 16, Loss: 0.037892330437898636
Epoch 17, Loss: 0.03834623098373413
Epoch 18, Loss: 0.039027608931064606


In [ ]:
model.eval()

X_test_t = torch.tensor(X_test, dtype=torch.float32)

preds = model(X_test_t).detach().numpy()

In [ ]:
# Only inverse target
y_test_actual = scaler.inverse_transform(
    np.concatenate([X_test[:, -1, :], y_test.reshape(-1,1)], axis=1)
)[:, -1]

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, preds)
print("MAE:", mae)

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, preds))
print("RMSE:", rmse)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(y_test, label="Actual")
plt.plot(preds, label="Predicted")
plt.legend()
plt.title("Demand Forecasting using LSTM")
plt.show()

In [ ]:
for i in range(10):
    print(f"Actual: {y_test[i]}, Predicted: {preds[i][0]}")